# Adaptive Laya Aggregator (Account A)

Downloads worker-0/1/2 artifacts, validates, merges, publishes merged checkpoint.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def _secret(name: str, default: str = "") -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or default
    except Exception:  # noqa: BLE001
        return os.environ.get(name, default)


REPO_URL = _secret("ADAPTIVE_LAYA_REPO")
GITHUB_TOKEN = _secret("GITHUB_TOKEN")
COMMIT = _secret("ADAPTIVE_LAYA_COMMIT")
METHOD = _secret("ADAPTIVE_LAYA_METHOD", "validation_weighted")
ROUND = _secret("ADAPTIVE_LAYA_ROUND", "0")
os.environ["ADAPTIVE_LAYA_METHOD"] = METHOD
os.environ["ADAPTIVE_LAYA_ROUND"] = ROUND

_repo_abs = "/kaggle/working/repo"
if REPO_URL and not Path(_repo_abs + "/configs/worker.yaml").exists():
    clone_url = REPO_URL
    if GITHUB_TOKEN and REPO_URL.startswith("https://github.com/"):
        clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    res = subprocess.run(["git", "clone", clone_url, _repo_abs], capture_output=True, text=True, check=False)
    if res.returncode != 0:
        print("CLONE STDOUT:", res.stdout)
        print("CLONE STDERR:", res.stderr)
os.chdir(_repo_abs)
if Path(_repo_abs + "/configs/worker.yaml").exists():
    if GITHUB_TOKEN and REPO_URL.startswith("https://github.com/"):
        subprocess.run(["git", "remote", "set-url", "origin",
                        REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")], check=False)
    subprocess.run(["git", "fetch", "origin"], check=False)
    if COMMIT:
        subprocess.run(["git", "checkout", COMMIT], check=True)
    else:
        subprocess.run(["git", "pull"], check=False)
req = Path(_repo_abs) / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
print("aggregator: method =", METHOD, "round =", ROUND)


In [ ]:
from pathlib import Path

import yaml

from scripts.kaggle_io import download, publish

cfg = yaml.safe_load(Path("configs/worker.yaml").read_text(encoding="utf-8"))
slugs = [w["artifact_slug"] for w in cfg["workers"]]
dirs = []
for s in slugs:
    try:
        d = download(s, dst=f"results/agg/{s.split('-')[-1]}")
        dirs.append(Path(d))
    except Exception as e:  # noqa: BLE001 - missing worker datasets must not abort aggregation
        print("skip", s, e)
print("workers available:", [str(d) for d in dirs])


In [ ]:
import subprocess
import sys

method = os.environ.get("ADAPTIVE_LAYA_METHOD", "validation_weighted")
out = "results/merged"
if len(dirs) == len(slugs):
    subprocess.run([sys.executable, "scripts/aggregate_adapters.py",
                    "--workers"] + [str(d) for d in dirs] +
                   ["--method", method, "--out", out], check=True)
    subprocess.run([sys.executable, "scripts/verify_checkpoint.py", out], check=True)
else:
    print(f"WARNING: only {len(dirs)}/{len(slugs)} worker datasets available - "
          "skipping aggregate/verify/publish (partial merges are not allowed)")

In [ ]:
from pathlib import Path

import yaml

cfg = yaml.safe_load(Path("configs/worker.yaml").read_text())
if Path("results/merged/manifest.json").exists():
    publish("results/merged", cfg["datasets"]["merged"], f"merged round {os.environ.get('ADAPTIVE_LAYA_ROUND','0')}")
    print("merged published")

In [ ]:
print("aggregation complete")